# SVAMP Evaluation: Saved SFT, GRPO, and DAPO Models

This standalone notebook evaluates your three saved adapters on a **new arithmetic word-problem benchmark**. It does not train models, generate teacher data, or modify existing model weights.

Default: **64 frozen random questions** from `ChilleD/SVAMP`, split `test` (300 rows before filtering). Each model gets the same questions, the same original student prompt, greedy decoding, a 2048-token output budget, and the same strict numeric verifier. Set TEST_N=0 to evaluate the full remaining test split. SVAMP is an additional benchmark, not necessarily harder than GSM8K.

The evaluation script is embedded; only this notebook needs to be uploaded separately. Your original project modules and saved adapters must remain on Drive. Models load sequentially in separate subprocesses. This notebook passed local syntax/helper checks; GPU execution has not been tested locally.

Sources: [SVAMP original project](https://github.com/arkilpatel/SVAMP), [dataset mirror and splits](https://huggingface.co/datasets/ChilleD/SVAMP).


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, subprocess
from pathlib import Path

PROJECT = Path('/content/drive/MyDrive/qwen3_1p7b_128_en')
ROOT = PROJECT/'runs'/'ds8b_qwen1p7b_128_001'
DAPO_RUN = 'dapo_001'
TEST_N = 64  # Set to 0 for all remaining SVAMP test questions (up to 300).
TOKENS = 2048
SEED = 42
NAME = f'svamp_{TEST_N if TEST_N else "all"}_seed{SEED}_b{TOKENS}'
OUT = ROOT/'external_eval'/NAME
STUDENT_CACHE = Path('/content/qwen1p7b_student_cache')

required = [PROJECT/'pipeline.py', PROJECT/'monitoring.py', PROJECT/'requirements.txt', ROOT/'manifest.json']
for adapter in [ROOT/'sft'/'final', ROOT/'grpo'/'final', ROOT/'dapo'/DAPO_RUN/'final']:
    required += [adapter/'adapter_config.json', adapter/'adapter_model.safetensors']
missing = [str(p) for p in required if not p.exists()]
assert not missing, 'Missing project files or adapters:\n'+'\n'.join(missing)
os.chdir(PROJECT)
print('Models found. New evaluation directory:', OUT)


Mounted at /content/drive
Models found. New evaluation directory: /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001/external_eval/svamp_64_seed42_b2048


## Dependencies

Use the original fixed project environment. The optional repair below removes unused Gradio/Diffusers only if pip check reports conflicts and the existing repair helper is available. It does not delete caches, datasets, or saved models. If Colab requests a restart, restart the session and rerun from the first cell. No teacher model is needed.


In [2]:
def stream_command(command, env=None, logfile=None):
    print('Running:', ' '.join(map(str, command)), flush=True)
    handle = open(logfile, 'a', encoding='utf-8') if logfile else None
    proc = subprocess.Popen(list(map(str, command)), cwd=PROJECT, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            if handle:
                handle.write(line)
                handle.flush()
        code = proc.wait()
        if code:
            raise RuntimeError(f'Command failed with exit code {code}. Read the output above. Log: {logfile}')
    except BaseException:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=20)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        raise
    finally:
        if handle:
            handle.close()

REPAIR_KNOWN_COLAB_CONFLICTS = True
stream_command([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT/'requirements.txt')])
check = subprocess.run([sys.executable, '-m', 'pip', 'check'],
                       capture_output=True, text=True)
print(check.stdout, check.stderr)
if check.returncode:
    repair = PROJECT/'repair_colab.py'
    if REPAIR_KNOWN_COLAB_CONFLICTS and repair.exists():
        stream_command([sys.executable, '-u', str(repair)])
    else:
        raise RuntimeError('Dependency conflicts remain. Inspect pip check output before proceeding.')
stream_command([sys.executable, '-m', 'pip', 'check'])
stream_command([sys.executable, '-c',
    'import importlib.metadata as m; from transformers import Trainer; from peft import PeftModel; '
    'from trl import GRPOTrainer; assert m.version("trl")=="0.23.1"; print("Training imports OK")'])


Running: /usr/bin/python3 -m pip install -r /content/drive/MyDrive/qwen3_1p7b_128_en/requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━

In [3]:
stream_command(['nvidia-smi'])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime.'
assert torch.cuda.is_bf16_supported(), 'The existing loader requires BF16 support.'
print('GPU:', torch.cuda.get_device_name())
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))


Running: nvidia-smi
Mon Oct 19 03:00:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------

## Prepare the embedded evaluator and freeze questions

The evaluator records the dataset revision, randomly selects the questions once, and checks normalized exact overlaps with your saved SFT/RL/development questions. Exact filtering cannot rule out semantic duplicates or pretraining overlap. Gold answers are used only for scoring, never included in prompts.


In [4]:
SVAMP_SOURCE = '"""Evaluate saved SFT, GRPO, and DAPO adapters on one frozen SVAMP test subset."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nfrom pipeline import STUDENT, prompt, read, write, normalize, uid, numeric, extract, correct, load_model, meta\nfrom monitoring import Progress\n\nDATASET = \'ChilleD/SVAMP\'\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1024*1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef select_questions(raw, forbidden, count, seed):\n    seen, rows = set(), []\n    for r in raw:\n        q = r[\'Body\'].strip()+\' \'+r[\'Question\'].strip()\n        key = normalize(q)\n        answer = str(r[\'Answer\'])\n        if numeric(answer) is None:\n            raise ValueError(f\'Non-numeric SVAMP gold answer: {answer}\')\n        if key not in forbidden and key not in seen:\n            seen.add(key)\n            rows.append({\'id\': uid(q), \'source_id\': r[\'ID\'], \'question\': q,\n                         \'answer\': answer, \'type\': r[\'Type\'], \'dataset\': \'svamp\'})\n    random.Random(seed).shuffle(rows)\n    if count > len(rows):\n        raise ValueError(f\'Requested {count} questions, but only {len(rows)} remain after deduplication.\')\n    return rows[:count] if count else rows\n\n\ndef freeze_questions(a, out):\n    data, manifest = out/\'questions.jsonl\', out/\'dataset.json\'\n    source_hashes = {name: sha(a.root/\'data\'/f\'{name}.jsonl\')\n                     for name in [\'sft_questions\', \'rl\', \'dev\'] if (a.root/\'data\'/f\'{name}.jsonl\').exists()}\n    if manifest.exists():\n        saved = json.loads(manifest.read_text())\n        if saved[\'requested_count\'] != a.n or saved[\'seed\'] != a.seed or saved[\'source_hashes\'] != source_hashes:\n            raise ValueError(\'Frozen dataset settings differ. Use a new --name.\')\n        if sha(data) != saved[\'questions_sha256\']:\n            raise ValueError(\'Frozen evaluation questions were modified.\')\n        return read(data)\n    if data.exists():\n        raise ValueError(\'Question file exists without a dataset manifest. Use a new --name.\')\n    from huggingface_hub import HfApi\n    from datasets import load_dataset\n    revision = HfApi().dataset_info(DATASET).sha\n    raw = load_dataset(DATASET, revision=revision, split=\'test\')\n    forbidden = set()\n    for name in source_hashes:\n        forbidden.update(normalize(r[\'question\']) for r in read(a.root/\'data\'/f\'{name}.jsonl\'))\n    rows = select_questions(raw, forbidden, a.n, a.seed)\n    write(data, rows)\n    manifest.write_text(json.dumps({\'dataset\': DATASET, \'split\': \'test\', \'revision\': revision,\n                                   \'requested_count\': a.n, \'actual_count\': len(rows), \'seed\': a.seed,\n                                   \'source_hashes\': source_hashes, \'questions_sha256\': sha(data),\n                                   \'dedup\': \'Normalized exact match only; semantic and pretraining overlap are unknown.\'},\n                                  indent=2), encoding=\'utf-8\')\n    return rows\n\n\ndef summarize(rows, stage, budget):\n    n = len(rows)\n    if not n:\n        raise ValueError(\'No predictions to summarize.\')\n    parsed = [extract(r[\'text\']) for r in rows]\n    return {\'stage\': stage, \'dataset\': \'svamp\', \'n\': n,\n            \'correct\': sum(correct(r[\'text\'], r[\'answer\']) for r in rows),\n            \'accuracy_percent\': 100*sum(correct(r[\'text\'], r[\'answer\']) for r in rows)/n,\n            \'answer_parse_percent\': 100*sum(x is not None and numeric(x) is not None for x in parsed)/n,\n            \'mean_tokens\': sum(r[\'tokens\'] for r in rows)/n,\n            \'mean_seconds\': sum(r[\'seconds\'] for r in rows)/n,\n            \'budget_hit_percent\': 100*sum(r[\'budget_hit\'] for r in rows)/n,\n            \'eval_budget\': budget}\n\n\ndef evaluate(a, questions, out):\n    import torch\n    from transformers import set_seed\n    set_seed(a.seed)\n    adapter = a.root/\'dapo\'/a.dapo_run/\'final\' if a.stage == \'dapo\' else a.root/a.stage/\'final\'\n    for name in [\'adapter_config.json\', \'adapter_model.safetensors\']:\n        if not (adapter/name).exists():\n            raise FileNotFoundError(adapter/name)\n    metadata = {\'stage\': a.stage, \'adapter_sha256\': sha(adapter/\'adapter_model.safetensors\'),\n                \'adapter_config_sha256\': sha(adapter/\'adapter_config.json\'),\n                \'base_revision\': meta(a.root)[\'model_revisions\'][STUDENT],\n                \'questions_sha256\': sha(out/\'questions.jsonl\'), \'decoding\': \'greedy\',\n                \'budget\': a.tokens, \'prompt_sha256\': hashlib.sha256(prompt(\'\').encode()).hexdigest(),\n                \'grader_sha256\': sha(Path(__file__).parent/\'pipeline.py\') if (Path(__file__).parent/\'pipeline.py\').exists()\n                                 else sha(Path(__import__(\'pipeline\').__file__))}\n    config_path = out/f\'{a.stage}.settings.json\'\n    predictions = out/f\'{a.stage}.predictions.jsonl\'\n    if config_path.exists() and json.loads(config_path.read_text()) != metadata:\n        raise ValueError(\'Saved evaluation settings/model differ. Use a new --name.\')\n    if predictions.exists() and not config_path.exists():\n        raise ValueError(\'Predictions exist without settings; use a new --name.\')\n    config_path.write_text(json.dumps(metadata, indent=2), encoding=\'utf-8\')\n    results = read(predictions) if predictions.exists() else []\n    expected = {q[\'id\']: (q[\'question\'], q[\'answer\']) for q in questions}\n    if len({r[\'id\'] for r in results}) != len(results) or any(\n            expected.get(r[\'id\']) != (r[\'question\'], r[\'answer\']) for r in results):\n        raise ValueError(\'Saved predictions do not match the frozen questions.\')\n    done = {r[\'id\'] for r in results}\n    progress = Progress(out, \'svamp_\'+a.stage, len(questions), initial=len(done))\n    progress.update(len(done))\n    if len(done) < len(questions):\n        model, tok = load_model(STUDENT, a.root, adapter=adapter)\n        model.eval()\n        with predictions.open(\'a\', encoding=\'utf-8\') as f:\n            for q in questions:\n                if q[\'id\'] in done:\n                    continue\n                inputs = tok(prompt(q[\'question\']), add_special_tokens=False, return_tensors=\'pt\').to(model.device)\n                torch.cuda.synchronize()\n                start = time.perf_counter()\n                with torch.inference_mode():\n                    ids = model.generate(**inputs, do_sample=False, max_new_tokens=a.tokens,\n                                         eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)\n                torch.cuda.synchronize()\n                new = ids[0, inputs.input_ids.shape[1]:]\n                text = tok.decode(new, skip_special_tokens=True)\n                row = {**q, \'text\': text, \'correct\': correct(text, q[\'answer\']), \'tokens\': len(new),\n                       \'seconds\': time.perf_counter()-start,\n                       \'budget_hit\': len(new) >= a.tokens and int(new[-1]) != tok.eos_token_id}\n                f.write(json.dumps(row, ensure_ascii=False)+\'\\n\')\n                f.flush()\n                done.add(q[\'id\'])\n                results.append(row)\n                progress.update(len(done), correct=row[\'correct\'], tokens=row[\'tokens\'])\n    summary = summarize(results, a.stage, a.tokens)\n    (out/f\'{a.stage}.summary.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\n    print(json.dumps(summary, indent=2))\n\n\ndef report(out):\n    summaries = [json.loads(p.read_text()) for stage in [\'sft\', \'grpo\', \'dapo\']\n                 if (p := out/f\'{stage}.summary.json\').exists()]\n    if not summaries:\n        print(\'No completed evaluations yet.\')\n        return\n    if len({(r[\'n\'], r[\'eval_budget\']) for r in summaries}) != 1:\n        raise ValueError(\'Evaluation counts or budgets differ across models.\')\n    with (out/\'comparison.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(summaries[0]))\n        writer.writeheader()\n        writer.writerows(summaries)\n    base_path = out/\'sft.predictions.jsonl\'\n    changes = []\n    if base_path.exists():\n        base = {r[\'id\']: correct(r[\'text\'], r[\'answer\']) for r in read(base_path)}\n        for stage in [\'grpo\', \'dapo\']:\n            path = out/f\'{stage}.predictions.jsonl\'\n            if not path.exists():\n                continue\n            rows = read(path)\n            if set(base) != {r[\'id\'] for r in rows}:\n                continue\n            improved = sum(not base[r[\'id\']] and correct(r[\'text\'], r[\'answer\']) for r in rows)\n            regressed = sum(base[r[\'id\']] and not correct(r[\'text\'], r[\'answer\']) for r in rows)\n            changes.append({\'stage\': stage, \'sft_wrong_to_right\': improved,\n                            \'sft_right_to_wrong\': regressed, \'net_correct_gain\': improved-regressed})\n    (out/\'paired_changes.json\').write_text(json.dumps(changes, indent=2), encoding=\'utf-8\')\n    print(\'Comparison:\', json.dumps(summaries, indent=2))\n    print(\'Paired changes relative to SFT:\', json.dumps(changes, indent=2))\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--root\', type=Path, required=True)\n    p.add_argument(\'--stage\', choices=[\'prepare\', \'sft\', \'grpo\', \'dapo\', \'report\'], required=True)\n    p.add_argument(\'--name\', default=\'svamp64_seed42\')\n    p.add_argument(\'--dapo-run\', default=\'dapo_001\')\n    p.add_argument(\'--n\', type=int, default=64, help=\'0 uses all remaining test questions (up to 300).\')\n    p.add_argument(\'--tokens\', type=int, default=2048)\n    p.add_argument(\'--seed\', type=int, default=42)\n    a = p.parse_args()\n    if a.n < 0 or a.tokens < 1:\n        p.error(\'n must be nonnegative and tokens must be positive.\')\n    for name in [a.name, a.dapo_run]:\n        if not name or Path(name).name != name or name in {\'.\', \'..\'}:\n            p.error(\'name/dapo-run must be simple directory names.\')\n    out = a.root/\'external_eval\'/a.name\n    out.mkdir(parents=True, exist_ok=True)\n    if a.stage == \'report\':\n        report(out)\n        return\n    questions = freeze_questions(a, out)\n    print(f\'Frozen SVAMP test questions: {len(questions)}; output: {out}\')\n    if a.stage != \'prepare\':\n        evaluate(a, questions, out)\n\n\nif __name__ == \'__main__\':\n    main()\n'
SCRIPT_DIR = Path('/content/svamp_standalone_runner')
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT = SCRIPT_DIR/'eval_svamp.py'
compile(SVAMP_SOURCE, str(SCRIPT), 'exec')
SCRIPT.write_text(SVAMP_SOURCE, encoding='utf-8')
OUT.mkdir(parents=True, exist_ok=True)
(OUT/'console_logs').mkdir(exist_ok=True)
env = dict(os.environ)
env['HF_HOME'] = str(STUDENT_CACHE)
env['PYTHONPATH'] = str(PROJECT)+os.pathsep+env.get('PYTHONPATH', '')
env['TOKENIZERS_PARALLELISM'] = 'false'
env['PYTHONUNBUFFERED'] = '1'

def evaluate_stage(stage):
    command = [sys.executable, '-u', str(SCRIPT), '--root', str(ROOT), '--stage', stage,
               '--name', NAME, '--dapo-run', DAPO_RUN, '--n', str(TEST_N),
               '--tokens', str(TOKENS), '--seed', str(SEED)]
    logfile = OUT/'console_logs'/f'{stage}_{time.time_ns()}.log'
    stream_command(command, env=env, logfile=logfile)

evaluate_stage('prepare')


Running: /usr/bin/python3 -u /content/svamp_standalone_runner/eval_svamp.py --root /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001 --stage prepare --name svamp_64_seed42_b2048 --dapo-run dapo_001 --n 64 --tokens 2048 --seed 42

Generating train split: 100%|██████████| 700/700 [00:00<00:00, 22689.08 examples/s]

Generating test split: 100%|██████████| 300/300 [00:00<00:00, 74862.64 examples/s]
Frozen SVAMP test questions: 64; output: /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001/external_eval/svamp_64_seed42_b2048


## Evaluate all three saved models

Each stage prints question progress, correctness, output length, elapsed time, and ETA. Predictions are saved after each question and reused on rerun if settings and model hashes match. Stop/restart does not require retraining. Do not change prompts or adapters within an existing evaluation directory.


In [5]:
for stage in ['sft', 'grpo', 'dapo']:
    print('\n'+'='*20+' '+stage.upper()+' '+'='*20, flush=True)
    evaluate_stage(stage)
evaluate_stage('report')



==================== SFT ====================
Running: /usr/bin/python3 -u /content/svamp_standalone_runner/eval_svamp.py --root /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001 --stage sft --name svamp_64_seed42_b2048 --dapo-run dapo_001 --n 64 --tokens 2048 --seed 42
Frozen SVAMP test questions: 64; output: /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001/external_eval/svamp_64_seed42_b2048
I0000 00:00:1792378872.082290    1357 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792378874.396238    1357 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[svamp_sft] [--------------------] 0/64 (0.0%) elapsed=00:00:00 ETA=estimating 
[load] Loading Qwen/Qwen3-1.7B-Base at revision ea980cb0a6

## Compare results

The first table reports strict answer accuracy, answer parse rate, output length, time, and generation-limit hits. The second table compares the *same questions*: SFT-wrong to RL-right versus SFT-right to RL-wrong.

If SFT still wins, the earlier result is reproduced on another benchmark. If RL wins here, gains may depend on the task distribution. Neither outcome alone proves a general improvement. With 64 questions, one answer is 1.56 percentage points; avoid tuning on these results repeatedly. Different GPUs invalidate direct timing comparisons across separate runs, but all models here use the same active runtime.


In [6]:
import pandas as pd
from IPython.display import display
print('SVAMP results:')
display(pd.read_csv(OUT/'comparison.csv').round(2))
print('Paired changes relative to SFT:')
display(pd.DataFrame(json.loads((OUT/'paired_changes.json').read_text())))
print('Saved question subset:', OUT/'questions.jsonl')
print('Prediction and result directory:', OUT)


SVAMP results:


,stage,dataset,n,correct,accuracy_percent,answer_parse_percent,mean_tokens,mean_seconds,budget_hit_percent,eval_budget
0,sft,svamp,64,53,82.81,100.00,248.95,18.34,3.12,2048
1,grpo,svamp,64,53,82.81,95.31,345.97,25.50,7.81,2048
2,dapo,svamp,64,53,82.81,98.44,406.30,29.95,10.94,2048


Paired changes relative to SFT:


,stage,sft_wrong_to_right,sft_right_to_wrong,net_correct_gain
0,grpo,2,2,0
1,dapo,3,3,0


Saved question subset: /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001/external_eval/svamp_64_seed42_b2048/questions.jsonl
Prediction and result directory: /content/drive/MyDrive/qwen3_1p7b_128_en/runs/ds8b_qwen1p7b_128_001/external_eval/svamp_64_seed42_b2048
